# Gene frequency from NLP-assisted literature extraction

Reconstructs the figure used to justify RT-qPCR target selection. Source data are immutable copies of the three extraction workbooks. The notebook documents the legacy procedure and creates English, publication-ready PDF and 600 dpi PNG outputs.

In [ ]:
from pathlib import Path
from collections import Counter
import ast
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
elif not (ROOT / 'data').exists() and (ROOT / 'article' / 'data').exists():
    ROOT = ROOT / 'article'
DATA = ROOT / 'data' / 'raw' / 'nlp_gene_selection'
FIGURES = ROOT / 'results' / 'figures'
OVERLEAF = ROOT / 'artigo_andre_tcc' / 'images'
FIGURES.mkdir(parents=True, exist_ok=True)
OVERLEAF.mkdir(parents=True, exist_ok=True)


## Load and parse the extraction tables

In the old notebook, the five fields returned by the language model were stored as a tuple-like string in `resposta` and parsed with a regular expression. Here `ast.literal_eval` is attempted first, with the original regular expression retained as a fallback. No exposure/path translations are needed for the gene-frequency calculation.

In [ ]:
FILES = {
    'Nicotine': DATA / 'resultados_nicotina.xlsx',
    'Heavy metals': DATA / 'resultados_metais.xlsx',
    'PAHs': DATA / 'resultados_pahs.xlsx',
}

def parse_response(value):
    if pd.isna(value):
        return [None] * 5
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, tuple) and len(parsed) == 5:
            return list(parsed)
    except (ValueError, SyntaxError):
        pass
    match = re.match(r"\('([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'([^']*)',\s*'(.*)'\)", str(value))
    return list(match.groups()) if match else [None] * 5

tables = {}
for group, path in FILES.items():
    df = pd.read_excel(path)
    parsed = pd.DataFrame(df['resposta'].map(parse_response).tolist(), index=df.index,
                          columns=['embryo', 'genes', 'exposure', 'pathway', 'summary'])
    tables[group] = pd.concat([df, parsed], axis=1)
    print(group, tables[group].shape, 'unparsed:', tables[group]['genes'].isna().sum())

## Reproduce and audit the historical counting rule

The legacy notebook split `genes` at commas, stripped whitespace, ignored `none`, and counted all tokens. It also excluded one nicotine row whose gene string was exactly `npy, egr1, bdnf, nrf2a`; that exclusion is retained explicitly below. For publication, repeated symbols within the same extraction record are counted once, so the plotted frequency represents records/articles rather than raw token repetitions. If DOI is populated, duplicate DOI–gene pairs are also collapsed.

In [ ]:
# Transformations that affected the gene column in the legacy notebook.
# Replacement order is preserved because the old helper built one alternation regex
# from the insertion-ordered dictionary and applied it to every string cell.
METAL_GENE_REPLACEMENTS = {
    'Zn': 'zinc', 'Pb (lead)': 'lead', 'As (arsenic)': 'arsenic',
    'Hg (mercury)': 'mercury', 'mercury (HgCl2)': 'mercury', 'Pb': 'lead',
    'copper (Cu2+), zinc (Zn2+)': 'copper, zinc', 'lead (Pb)': 'lead',
    'Cd (cadmium)': 'cadmium', 'arsenic (As)': 'arsenic',
    'cadmium chloride (CdCl2)': 'cadmium', 'silica nanoparticles (SiNPs)': 'silica',
    'lead acetate': 'lead', 'chromium (Cr)': 'chromium',
    'mercury chloride': 'mercury', 'cadmium (Cd)': 'cadmium',
    'copper (Cu)': 'copper',
    'zinc (Zn2+), copper (Cu2+), cadmium (Cd2+)': 'zinc, copper, cadmium',
    'copper (Cu2+), cadmium (Cd2+)': ' copper, cadmium',
    'zinc (Zn)': 'zinc', 'copper (Cu2+), mercury (Hg2+)': 'copper, mercury',
    'zinc chloride': 'zinc', 'Hg': 'mercury', 'Cadmium (Cd2+)': 'cadmium',
    'Silver nitrate (AgNO3)': 'silver', 'Lanthanum (La)': 'lanthanum',
    'mercuric chloride (HgCl2)': 'mercury',
    'mercury compounds (HgCl2, MeHg)': 'mercury', 'As': 'arsenic',
    'SiNPs and PbAc (lead acetate)': 'silica, lead', 'iron (Fe)': 'iron',
    'Cu (copper)': 'copper', 'Cu': 'copper', 'paraquat (PQ)': 'paraquat',
    'copper sulfate': 'copper', 'mercury (Hg)': 'mercury',
    'lead or mercury': 'lead, mercury', '2+': ' ', 'Cd': 'cadmium',
    'manganese (Mn)': 'manganese', 'lead and cadmium': 'lead, cadmium',
    'heavy metals': 'cadmium, mercury, copper', 'Praseodymium (Pr)': 'praseodymium',
    'and': ',', 'mt2 (implied by metallothionein promoter)': 'mt2',
    '-aminolevulinic acid dehydratase (ALA-D)': 'ALA-D',
    'superoxide dismutase (SOD), catalase (CAT), glutathione peroxidase (GPx), glutathione-S-transferase (GST)': 'SOD, CAT, GPx, GST',
    'glutathione-S-transferase (GST)': 'GST', 'glutathione peroxidase (GPx)': 'GPx',
    'superoxide dismutase (SOD)': 'SOD', 'catalase (CAT)': 'CAT',
    'glutathione reductase (GR)': 'GR', 'stress response': 'oxidative stress',
    'oxidative stress response': 'oxidative stress', 'neurogenesis': 'neurodevelopment',
    'oxidative defense': 'oxidative stress',
    'central nervous system neurodevelopment': 'neurodevelopment',
    'mercury+': 'mercury', 'AgNPs': 'silver', 'AgNO3': 'silver',
    'arsenic3+': 'arsenic', 'lead)': 'lead', 'Ba': 'barium', 'Se': 'selenium'
}
def legacy_replace(value, replacements):
    if not isinstance(value, str):
        return value
    pattern = re.compile('|'.join(map(re.escape, replacements)))
    return pattern.sub(lambda match: replacements[match.group()], value)

tables['Heavy metals']['genes'] = tables['Heavy metals']['genes'].map(
    lambda value: legacy_replace(value, METAL_GENE_REPLACEMENTS))
tables['PAHs']['genes'] = tables['PAHs']['genes'].str.lower()

EXCLUDED_NICOTINE_GENE_STRING = 'npy, egr1, bdnf, nrf2a'  # exact legacy exclusion
tables['Nicotine'] = tables['Nicotine'].loc[tables['Nicotine']['genes'] != EXCLUDED_NICOTINE_GENE_STRING].copy()

def gene_tokens(value):
    if pd.isna(value) or str(value).strip().lower() in {'', 'none'}:
        return []
    return [token.strip() for token in str(value).split(',') if token.strip()]

def legacy_counts(df):
    return Counter(gene for value in df['genes'] for gene in gene_tokens(value))

def article_counts(df):
    pairs = set()
    doi_col = next((c for c in ['DOI', 'doi'] if c in df.columns), None)
    for idx, row in df.iterrows():
        article = str(row[doi_col]).strip().lower() if doi_col and pd.notna(row[doi_col]) else f'row:{idx}'
        for gene in set(gene_tokens(row['genes'])):
            pairs.add((article, gene))
    return Counter(gene for _, gene in pairs)

audit_rows = []
counts = {}
for group, df in tables.items():
    old, audited = legacy_counts(df), article_counts(df)
    counts[group] = audited
    for gene in sorted(set(old) | set(audited)):
        audit_rows.append({'group': group, 'gene': gene, 'legacy_token_count': old[gene],
                           'distinct_article_count': audited[gene], 'difference': audited[gene] - old[gene]})
audit = pd.DataFrame(audit_rows)
audit.to_csv(ROOT / 'data' / 'processed' / 'nlp_gene_frequency_audit.csv', index=False)
audit.sort_values(['group', 'distinct_article_count'], ascending=[True, False]).groupby('group').head(10)

## Publication figure

The five most frequently reported genes are shown for each contaminant group. PDF is the preferred vector format for Overleaf; PNG is exported at 600 dpi for previews and raster-only workflows.

In [ ]:
TOP_K = 5
COLORS = {'Nicotine': '#4477AA', 'Heavy metals': '#EE6677', 'PAHs': '#228833'}
fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
group_centers, group_labels = [], []
cursor = 0
for group in FILES:
    top = counts[group].most_common(TOP_K)
    genes, frequencies = zip(*top) if top else ([], [])
    x = np.arange(len(genes)) + cursor
    bars = ax.bar(x, frequencies, color=COLORS[group], edgecolor='black', linewidth=0.9)
    for bar, gene in zip(bars, genes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8, gene,
                ha='center', va='bottom', rotation=62, fontsize=14, fontstyle='italic')
    group_centers.append(float(x.mean()))
    group_labels.append(group)
    cursor = int(x[-1]) + 3
ax.set_xticks(group_centers, group_labels, fontsize=16)
ax.set_xlabel('Dataset', fontsize=17, labelpad=8)
ax.set_ylabel('Frequency (articles)', fontsize=17)
ax.tick_params(axis='y', labelsize=14)
ax.grid(axis='y', linestyle='--', linewidth=0.7, alpha=0.35)
ax.set_axisbelow(True)
ax.set_ylim(0, max(max(counts[group].values()) for group in FILES) * 1.17)
for suffix, kwargs in [('pdf', {}), ('png', {'dpi': 600})]:
    for destination in [FIGURES, OVERLEAF]:
        fig.savefig(destination / f'genes_selected_from_nlp.{suffix}', bbox_inches='tight', **kwargs)
plt.show()